This notebook is a WIP of the fine-tuning of ChemBERTa on the PolyMetriX dataset

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")

PolyMetriX dataset

In [3]:
from polymetrix.datasets import CuratedGlassTempDataset

dataset = CuratedGlassTempDataset()

# Extract one molecule from the dataset
psmiles = dataset.psmiles[0]
print(psmiles)
tokens = tokenizer(psmiles, return_tensors="pt")
print(tokens)
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

[*]#C[SiH2]C#Cc1cccc(C#[*])c1
{'input_ids': tensor([[  0,  63,  14,  65,   7,  39,  63,  55,  77,  44,  22,  65,  39,   7,
         267,  21, 276,  12,  39,   7,  63,  14,  65,  13,  71,  21,   2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]])}
['<s>', '[', '*', ']', '#', 'C', '[', 'S', 'i', 'H', '2', ']', 'C', '#', 'Cc', '1', 'cccc', '(', 'C', '#', '[', '*', ']', ')', 'c', '1', '</s>']


Need to replace "[*]" to "C" for ChemBERTa

In [4]:
from datasets import Dataset

hf_dataset = Dataset.from_dict({
    "smiles": [psmiles.replace("[*]", "C") for psmiles in dataset.psmiles.tolist()],
    "labels": dataset._labels.flatten().tolist()
})

def tokenize(batch):
    return tokenizer(batch["smiles"], truncation=True, max_length=512)

hf_dataset = hf_dataset.map(tokenize, batched=True)
print(hf_dataset[0].keys(), hf_dataset[0]['smiles'])

Map:   0%|          | 0/7367 [00:00<?, ? examples/s]

dict_keys(['smiles', 'labels', 'input_ids', 'attention_mask']) C#C[SiH2]C#Cc1cccc(C#C)c1


In [5]:
seed = 42
# split 80% train, 20% temp
split1 = hf_dataset.train_test_split(test_size=0.2, seed=seed)

# split 20% temp to 10% val, 10% test
split2 = split1["test"].train_test_split(test_size=0.5, seed=seed)

print(len(split1["train"]) + len(split2["train"]) + len(split2["test"]), len(hf_dataset))

7367 7367


Remove the smiles string from the datasets

In [6]:
cols_to_remove = ["smiles"]
split1["train"] = split1["train"].remove_columns(cols_to_remove)
split2["train"] = split2["train"].remove_columns(cols_to_remove)
split2["test"] = split2["test"].remove_columns(cols_to_remove)

split1["train"].set_format("torch")
split2["train"].set_format("torch")
split2["test"].set_format("torch")

Load the ChemBERTa model for SequenceClassification with num_labels = 1 -> regression

In [7]:
model_chemberta = AutoModelForSequenceClassification.from_pretrained(
    "seyonec/ChemBERTa-zinc-base-v1",
    num_labels=1
)

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Use RMSE and MAE metrics like for MolProPreNet

In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    mae = mean_absolute_error(labels, predictions)
    return {"rmse": rmse, "mae": mae}

Set up the Trainer. Use data_collator in order to have equal token sequences

In [9]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./chemberta_tg",
    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model_chemberta,
    args=training_args,
    train_dataset=split1["train"],
    eval_dataset=split2["train"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

Train the model

In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rmse,Mae
1,No log,172467.468750,415.292028,399.602783
2,175615.952000,168124.484375,410.029858,394.131012
3,167822.048000,164190.921875,405.204790,389.108856
4,167822.048000,160555.781250,400.694124,384.409332
5,163681.728000,157191.515625,396.473852,380.008270
6,157901.264000,154084.250000,392.535667,375.897614
7,155585.232000,151224.250000,388.875623,372.073853
8,155585.232000,148603.640625,385.491427,368.535461
9,150033.552000,146215.609375,382.381477,365.281158
10,146233.024000,144054.218750,379.544752,362.310638


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=7380, training_loss=148384.15989159892, metrics={'train_runtime': 440.9361, 'train_samples_per_second': 267.295, 'train_steps_per_second': 16.737, 'total_flos': 3090395066956470.0, 'train_loss': 148384.15989159892, 'epoch': 20.0})

Evaluation on the test set

In [11]:
prediction = trainer.predict(split2["test"])
rmse_loss = prediction.metrics["test_rmse"]
mae_loss = prediction.metrics["test_mae"]
print(f"RMSE test: {rmse_loss:.4f} | MAE test: {mae_loss:.4f}")

RMSE test: 367.2469 | MAE test: 350.3477


Total number of parameters in ChemBERTa

In [12]:
print(sum(p.numel() for p in model_chemberta.parameters() if p.requires_grad))

44104705
